My code

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 135.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gensim]2m2/3 [gensim]


In [2]:
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
"""
Module 1: Dataset Loading
-------------------------
This module loads the Steam reviews dataset from a local CSV file.
The dataset contains user-written reviews along with sentiment labels.
"""

# Load Steam reviews dataset
df = pd.read_csv("./dataset.csv")
df.head()

,review_text,review_score
0,Ruined my life.,1
1,This will be more of a ''my experience with th...,1
2,This game saved my virginity.,1
3,• Do you like original games? • Do you like ga...,1
4,"Easy to learn, hard to master.",1


In [5]:
"""
Module 2: Column Selection and Renaming
---------------------------------------
This module extracts only the columns relevant for sentiment analysis:
- review_text: the textual content of the review
- review_score: the sentiment label (1 = positive, 0 = negative)

The columns are renamed for consistency with the rest of the pipeline.
"""

# Keep only relevant columns
df = df[['review_text', 'review_score']]

# Rename columns for consistency
df.columns = ['text', 'label']

"""
Module 3: Label Conversion and Data Cleaning
--------------------------------------------
This module ensures sentiment labels are stored as integers and
removes invalid or empty reviews to improve model reliability.
"""

# Convert label to integer (safety)
df['label'] = df['label'].astype(int)

# Remove missing or empty reviews
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.strip() != ""]

# Check class balance
df['label'].value_counts()

label
 1    42028
-1     7833
Name: count, dtype: int64

In [6]:
"""
Module 4: Train-Test Split
--------------------------
This module separates the dataset into training and testing subsets.
Stratified sampling is used to preserve class balance across splits.
"""

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
"""
Module 5: Text Vectorization (TF-IDF)
-------------------------------------
This module converts raw text reviews into numerical feature vectors
using TF-IDF with unigrams and bigrams.

Key parameters:
- max_features: limits vocabulary size for efficiency
- stop_words: removes common English words
"""


vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words='english',
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [8]:
"""
Module 6: Model Training
-----------------------
This module trains a Logistic Regression classifier on the TF-IDF
feature vectors to learn sentiment patterns.
"""

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [9]:
"""
Module 7: Model Evaluation
-------------------------
This module evaluates the trained model on unseen test data.
Metrics reported include:
- Accuracy
- Precision, Recall, F1-score
- Confusion Matrix
"""

# Predict on test data
y_pred = model.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# Detailed performance metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Accuracy: 0.8946

Classification Report:
              precision    recall  f1-score   support

          -1       0.77      0.47      0.58      1567
           1       0.91      0.97      0.94      8406

    accuracy                           0.89      9973
   macro avg       0.84      0.72      0.76      9973
weighted avg       0.89      0.89      0.88      9973


Confusion Matrix:
[[ 735  832]
 [ 219 8187]]


In [10]:
"""
Module 8: Sentiment Prediction Function
---------------------------------------
This module defines a helper function that predicts the sentiment
of a single, unseen review using the trained vectorizer and model.
"""

def predict_sentiment(review_text):
    review_vector = vectorizer.transform([review_text])
    prediction = model.predict(review_vector)[0]
    return "Positive" if prediction == 1 else "Negative"

In [11]:
"""
Module 9: Example Predictions
-----------------------------
This module demonstrates sentiment prediction on new,
previously unseen Steam game reviews.
"""

example_review_1 = "This game is absolutely amazing. The mechanics are smooth and the story is engaging."
example_review_2 = "The game crashes constantly and the controls are terrible."

print(example_review_1, "→", predict_sentiment(example_review_1))
print(example_review_2, "→", predict_sentiment(example_review_2))

This game is absolutely amazing. The mechanics are smooth and the story is engaging. → Positive
The game crashes constantly and the controls are terrible. → Negative


Original Code

In [12]:
"""
 * Module: Imports
 * ----------------
 * This module imports all external libraries required for the sentiment
 * analysis pipeline.
 *
 * Libraries used:
 * - gensim.models.Word2Vec: Used to train word embeddings from text data.
 * - numpy: Provides numerical and vector operations.
 * - re: Used for regular expression-based text cleaning.
 """


from gensim.models import Word2Vec
import numpy as np
import re

In [13]:
# ----------------------
# Dataset
# ----------------------
sentences = [
    ("The movie was amazing and full of heart", 1),
    ("A boring plot with terrible acting", 0),
    ("I loved the characters but hated the ending", 0),
    ("The film was not good at all", 0),
    ("Surprisingly fun and well written", 1),
    ("I expected more it was disappointing", 0),
    ("Absolutely fantastic experience", 1),
    ("The story was dull and predictable", 0)
]

In [14]:
# ----------------------
# Preprocessing
# ----------------------
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()

In [15]:
corpus = [tokenize(text) for text, _ in sentences]
print(corpus)

[['the', 'movie', 'was', 'amazing', 'and', 'full', 'of', 'heart'], ['a', 'boring', 'plot', 'with', 'terrible', 'acting'], ['i', 'loved', 'the', 'characters', 'but', 'hated', 'the', 'ending'], ['the', 'film', 'was', 'not', 'good', 'at', 'all'], ['surprisingly', 'fun', 'and', 'well', 'written'], ['i', 'expected', 'more', 'it', 'was', 'disappointing'], ['absolutely', 'fantastic', 'experience'], ['the', 'story', 'was', 'dull', 'and', 'predictable']]


In [16]:
# ----------------------
# Train Word2Vec
# ----------------------
model = Word2Vec(corpus, vector_size=50, window=4, min_count=1, sg=1)

In [17]:
# ----------------------
# Sentence vector
# ----------------------
def sentence_vector(tokens):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

In [18]:
X = np.array([sentence_vector(tokens) for tokens in corpus])
y = np.array([label for _, label in sentences])

In [19]:
# ----------------------
# Simple sentiment prototypes
# ----------------------
pos_vec = np.mean(X[y == 1], axis=0)
neg_vec = np.mean(X[y == 0], axis=0)

In [20]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def predict(sentence):
    v = sentence_vector(tokenize(sentence))
    return "positive" if cosine(v, pos_vec) > cosine(v, neg_vec) else "negative"

In [21]:
# ----------------------
# Try it
# ----------------------
tests = [
    "great acting and wonderful story",
    "painfully slow and boring",
    "not bad but not great",
    "I loved the visuals"
]

for t in tests:
    print(t, "→", predict(t))

great acting and wonderful story → negative
painfully slow and boring → positive
not bad but not great → negative
I loved the visuals → negative
